# 품질 자료 검색 · 요약 챗봇 — Gradio + OpenAI

**현장 데이터 디지털화 과정 2차 · Day05 챗봇(코랩) 요청 — 품질 자료 검색 · 요약**

사내 품질 문서(품질매뉴얼 · 절차서 · 부적합 사례집)를 올리면, 질문에 **3~5문장 요약**으로 답하고
**근거 문서명과 항목 번호**(예: 「검사 절차서 SOP-QC-012, 4.2항」)를 붙이는 챗봇입니다.
문서에서 찾지 못하면 「확인되지 않습니다」라고 답하고, 문서끼리 내용이 다르면 둘 다 보여 줍니다.

### 시작하는 방법
1. 메뉴 **런타임 → 모두 실행**. ① 설치는 1분쯤 걸립니다.
2. ⑤ 셀 아래 화면에서 **「예제 문서로 연습하기」** → 「수입검사 시료는 몇 개 뽑아?」를 물어봅니다(문서 간 차이가 나오는 질문).
3. 내 문서로 바꿀 때는 왼쪽에 파일을 올리고 **「문서 읽기」**(pdf · docx · xlsx · pptx · hwpx · txt).
4. OpenAI 키가 없어도 돌아갑니다 — 그때는 요약 대신 **가장 가까운 항목**을 원문 그대로 보여 줍니다.

## 요청서를 어떻게 반영했나

| 요청 | 이 노트북에서 |
|---|---|
| 매뉴얼 · 절차서 · 부적합 사례집에서 질문으로 찾기 | 파일을 **항목(4.2 · 제5조 같은 조항 제목) 단위**로 나눠 두고, 질문과 가까운 항목을 코랩 안에서 찾습니다 |
| 요약 답변 3~5문장 | AI 에게 3~5문장으로 요약하게 하고, 넘으면 다시 요청하지 않고 그대로 보여 줍니다(문장 수는 화면 아래 표시) |
| 근거 문서명 · 항목 번호 | 문서 번호(SOP-QC-012 등)는 **파일 이름이나 문서 첫머리에서**, 항목 번호는 **조항 제목에서** 파이썬이 읽어 붙입니다. 사례집(엑셀)은 사례 번호가 항목 번호입니다 |
| 못 찾으면 「확인되지 않습니다」 | ① 가까운 항목이 거의 없으면 AI 를 부르지 않고 바로 ② AI 도 문서에 없다고 판단하면 같은 문장으로 |
| 추측 · 지어내기 금지 | AI 가 적은 근거 원문이 실제 문서에 있는지 **코드가 다시 대조**해, 다르면 ⚠️ 표시 |
| 문서 간 내용이 다르면 둘 다 | AI 가 차이를 「문서 간 차이」 칸에 따로 적고, 화면에 두 근거를 나란히 보여 줍니다 |
| 개인정보 · 기밀 질문은 담당 부서 안내 | 질문에 개인정보 · 기밀 낱말(연락처 · 주민번호 · 급여 · 단가 · 원가 · 기밀 …)이나 전화번호 모양이 있으면 **검색도 AI 도 하지 않고** 담당 부서 확인을 안내합니다 |

**요청과 다르게 한 것 · 못 한 것과 이유**

| 요청 | 이렇게 했습니다 | 이유 |
|---|---|---|
| 「지난해 동일 불량」 | 사례집에서 **불량 이름 · 공정**으로 찾습니다. 「지난해」는 사례의 발생일 칸으로 AI 가 확인합니다 | 「동일 불량」이 무엇인지는 질문에 불량 이름이 있어야 찾을 수 있습니다(예: 「작년 버 불량 원인과 조치는?」) |
| 항목 번호 | 조항 제목이 「4.2」 · 「4.2.1」 · 「제5조」 · 「5)」 모양일 때 읽습니다 | 제목에 번호가 없는 문서는 「문단 3~8」처럼 위치로 표시됩니다 |
| 스캔 PDF | 읽지 않습니다(글자가 없다고 알림) | 그림 속 글자는 OCR 이 필요합니다 — Day05 학습 도우미(ex2) 노트북 방식 참고 |
| 옛 형식 .hwp · .doc | 읽지 않고 안내 | HWPX · DOCX · PDF 로 저장해 올려 주세요 |

> ⚠️ **사내 문서 반출 확인**: 키를 넣으면 **질문과 가까운 항목 몇 개(전체 문서가 아님)** 가 OpenAI 로 갑니다. 사내 품질 문서를 외부 AI 에 보내도 되는지 먼저 회사 규정을 확인하세요. 키 없이 쓰면 문서는 코랩 밖으로 나가지 않습니다.

## 이 챗봇을 AI 에게 만들어 달라고 할 때 쓴 프롬프트 (보완판)

요청서(목적 · 사용자 · 입력 · 출력 · 제약)는 그대로 두고, 코랩에서 한 번에 돌아가도록 **[기술 조건]** 을 덧붙였습니다.

```text
너는 파이썬 Gradio 개발자야. 구글 코랩에서 바로 실행되는 「품질 자료 검색 · 요약 챗봇」을 만들어줘.

[목적 · 사용자] 품질 관리 담당자 · 감사 · 인증 준비 담당자가 사내 품질 문서(매뉴얼 · 절차서 · 부적합 사례집)에서 필요한 내용을 질문으로 빠르게 찾게 해줘.

[출력]
- 질문에 대한 요약 답변 3~5문장
- 근거 문서명과 항목 번호(예: 「검사 절차서 SOP-QC-012, 4.2항」)
- 근거를 찾지 못하면 「확인되지 않습니다」

[제약]
- 문서에 없는 내용은 추측하거나 지어내지 마.
- 문서 간 내용이 다르면 둘 다 제시하고 차이를 알려줘.
- 개인정보 · 기밀 관련 질문은 담당 부서 확인을 안내해줘.

[기술 조건]
- 파일은 형식에 맞는 라이브러리(pypdf · python-docx · openpyxl · python-pptx)로 읽고, 「4.2」 · 「제5조」 같은 조항 제목에서 항목 번호를 읽어 조각마다 붙여줘. 문서 번호는 파일 이름이나 첫머리에서 찾아줘.
- 검색은 코랩 안에서(scikit-learn TF-IDF) 하고, 가까운 항목만 AI 에게 보내줘. 점수가 아주 낮으면 AI 를 부르지 말고 「확인되지 않습니다」.
- 개인정보 · 기밀 질문은 검색 전에 파이썬이 먼저 걸러줘.
- AI 답은 JSON(answer · found · sources · conflicts)으로 받고, 근거 원문이 실제 문서에 있는지 코드로 대조해줘.
- 최신 Gradio 6. 대화 기록은 {"role", "content"} 형식, 튜플 형식은 쓰지 마. 돌려주는 값의 개수는 늘 outputs 개수와 같게.
- OpenAI 는 openai 1.x 의 OpenAI() 와 [gpt-4.1-mini]. 키가 없어도 가까운 항목을 원문 그대로 보여 주게 해줘.
- 요청대로 할 수 없는 것은 노트북 맨 위에 「못 한 것과 이유」 표로 적어줘.

[대상] 개발을 처음 해 보는 [품질경영팀 실무자]가 씁니다. 화면 글자는 모두 한국어로 짧게.
```

## ① 설치 (1분쯤)

In [ ]:
!pip install -q -U gradio openai pypdf python-docx openpyxl python-pptx scikit-learn
import gradio, openai
print("Gradio", gradio.__version__, "· openai", openai.__version__)

## ② 예제 문서 만들기 (연습용 · 모두 지어낸 내용)
품질매뉴얼 · 수입검사 절차서 · 공정검사 절차서 · 부적합 사례집 4개를 만듭니다. **수입검사 시료 수가 문서마다 다르게** 적혀 있어 「문서 간 차이」를 연습할 수 있습니다.

In [ ]:
import os, docx, openpyxl
D = os.environ.get("SAMPLE_DIR", "/content/품질문서_예제")
os.makedirs(D, exist_ok=True)

def make_docx(path, title, paras):
    d = docx.Document()
    d.add_heading(title, 1)
    for p in paras:
        d.add_paragraph(p)
    d.save(path)

make_docx(f"{D}/품질매뉴얼_QM-001.docx", "품질매뉴얼 QM-001 (예제 · 지어낸 내용)", [
    "문서번호: QM-001  개정: 5판",
    "7.1 수입검사",
    "구매품은 입고 시 수입검사를 한다. 로트마다 시료 5개를 뽑아 외관과 치수를 확인한다.",
    "세부 방법은 검사 절차서 SOP-QC-012 를 따른다.",
    "7.2 공정검사",
    "공정검사는 초물 · 중물 · 종물 검사로 하며 세부 방법은 SOP-QC-015 를 따른다.",
    "8.3 부적합품 관리",
    "부적합품은 빨간색 표찰을 붙여 격리 구역에 두고, 24시간 안에 부적합 보고서를 작성한다.",
    "8.5 시정조치",
    "같은 부적합이 3개월 안에 2회 이상 생기면 시정조치 요구서(CAR)를 발행하고 원인 분석을 한다.",
])

make_docx(f"{D}/검사절차서_SOP-QC-012.docx", "수입검사 절차서 SOP-QC-012 (예제 · 지어낸 내용)", [
    "문서번호: SOP-QC-012  개정: 3판",
    "4.1 적용 범위",
    "외부에서 구매한 원자재 · 부품 · 외주 가공품에 적용한다.",
    "4.2 검사 기준",
    "시료 수는 로트 크기에 따라 정한다. 500개 이하 8개, 501~3,200개 13개, 3,201개 이상 20개를 뽑는다.",
    "외관은 긁힘 · 녹 · 버(burr)가 없어야 하고, 치수는 도면 공차 안에 있어야 한다.",
    "4.3 판정",
    "시료 가운데 불합격이 1개라도 있으면 그 로트는 불합격으로 하고 공급자에게 통보한다.",
    "4.4 기록",
    "검사 결과는 수입검사 성적서(양식 QF-012-1)에 기록하고 3년간 보관한다.",
])

make_docx(f"{D}/공정검사절차서_SOP-QC-015.docx", "공정검사 절차서 SOP-QC-015 (예제 · 지어낸 내용)", [
    "문서번호: SOP-QC-015  개정: 2판",
    "5.1 초물 검사",
    "작업 시작 · 금형 교체 · 작업자 교대 후 첫 제품 3개를 검사하고 합격해야 양산을 시작한다.",
    "5.2 중물 검사",
    "2시간마다 2개를 뽑아 치수와 외관을 확인한다.",
    "5.3 종물 검사",
    "작업 종료 시 마지막 제품 3개를 검사해 금형 상태를 확인한다.",
    "5.4 이상 발생 시 조치",
    "검사 불합격이면 즉시 라인을 멈추고 직전 합격 시점 이후 생산품을 전수 검사한다.",
])

wb = openpyxl.Workbook(); ws = wb.active; ws.title = "부적합사례"
ws.append(["사례번호", "발생일", "공정", "부적합 내용", "원인", "조치", "재발 방지"])
for r in [
    ["NC-2025-003", "2025-03-14", "프레스", "버 과다(burr) — 고객 반품 120개", "금형 날 마모", "금형 연마 · 전수 선별", "금형 타수 관리 기준 5만 타로 변경"],
    ["NC-2025-007", "2025-06-02", "용접", "용접 비드 기공", "보호 가스 유량 부족", "가스 유량 조정 · 재용접", "작업 시작 전 유량 점검표 추가"],
    ["NC-2025-011", "2025-09-21", "도장", "도막 두께 미달", "스프레이 노즐 막힘", "노즐 교체 · 재도장", "주간 노즐 청소 일정 수립"],
    ["NC-2025-014", "2025-11-05", "프레스", "버 과다(burr) — 공정 내 발견 40개", "금형 날 마모(타수 4.8만)", "금형 연마", "타수 기준 4만 타로 강화 검토"],
]:
    ws.append(r)
wb.save(f"{D}/부적합사례집_NC-2025.xlsx")

os.environ["SAMPLE_DIR"] = D
print(sorted(os.listdir(D)))

## ③ (선택) 구글 드라이브 폴더 연결 — 내 품질 문서를 드라이브에서 읽을 때
파일을 화면에서 직접 올릴 거라면 건너뛰세요.

In [ ]:
# (선택) 구글 드라이브의 품질 문서 폴더에서 읽을 때만 실행하세요
from google.colab import drive
drive.mount("/content/drive")
import os
os.environ["DOC_FOLDER"] = "/content/drive/MyDrive/품질문서"   # ← 내 폴더 경로로 바꾸기
print("폴더:", os.environ["DOC_FOLDER"], "· 있음" if os.path.isdir(os.environ["DOC_FOLDER"]) else "· 아직 없음")

## ④ API 키 불러오기
코랩 왼쪽 🔑 보안 비밀에 `OPENAI_API_KEY` 를 넣고 노트북 액세스를 켜 두면 읽어 옵니다. 없으면 키 없이(원문 항목 보기) 돌아갑니다.

In [ ]:
import os, time
os.environ["TZ"] = "Asia/Seoul"      # 코랩 시계는 UTC — 한국 시각으로
time.tzset()
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY") or ""
except Exception:
    pass
print("OPENAI_API_KEY", "✅ 있음 — 요약 답변" if os.environ.get("OPENAI_API_KEY") else "— 없음 — 가까운 항목을 원문 그대로 보여 줍니다(화면 칸에 넣어도 됩니다)")

## ⑤ 챗봇 실행
아래에 화면이 뜹니다. `https://….gradio.live` 를 누르면 새 창에서 크게 열립니다. 멈추려면 ■, 코드를 고쳤으면 이 셀만 다시 실행하세요.

In [ ]:
# ============================================================
# 품질 자료 검색 · 요약 챗봇 (Gradio + OpenAI)
#  - 문서를 「항목(조항)」 단위로 나눠 문서번호 · 항목 번호를 붙여 둡니다.
#  - 검색은 코랩 안(TF-IDF), AI 에게는 가까운 항목 몇 개만 보냅니다.
#  - 개인정보 · 기밀 질문은 검색 전에 걸러 담당 부서 확인을 안내합니다.
# ============================================================
import os, re, json, html, zipfile, inspect
import gradio as gr

TOP_K = 6              # AI 에게 보내는 항목 수
MIN_SCORE = 0.03       # 이보다 가까운 항목이 없으면 AI 를 부르지 않고 「확인되지 않습니다」
MAX_CHARS = 1500       # 항목 하나가 이보다 길면 나눠 둡니다(같은 항목 번호 유지)
NOT_FOUND = "확인되지 않습니다. 올린 문서에서 이 질문의 근거를 찾지 못했습니다."
CONTACT = "품질경영팀 문서 관리 담당 또는 정보보안 담당 부서"   # ← 회사에 맞게 바꾸기

# ---------- 1. 문서 번호 · 항목 번호 읽기 ----------
# 앞뒤가 영문 · 숫자가 아니면 됩니다(「검사절차서_SOP-QC-012」처럼 _ 나 한글 뒤에 붙어 있어도 찾도록 \b 를 쓰지 않습니다)
DOC_ID = re.compile(r"(?<![A-Za-z0-9])([A-Z]{2,5}(?:-[A-Z]{2,5})?-\d{2,4}(?:-\d{1,3})?)(?![0-9])")
# 「4.2 검사 기준」 「4.2.1」 「제5조(목적)」 「5) 판정」 처럼 줄 맨 앞의 번호를 조항 제목으로 봅니다
CLAUSE = re.compile(r"^\s*(제\s*\d+\s*조(?:의\s*\d+)?|\d{1,2}(?:\.\d{1,2}){1,3}\.?|\d{1,2}\)|\d{1,2}\.)\s*(\S.*)?$")

def doc_id_of(filename, text):
    m = DOC_ID.search(filename) or DOC_ID.search(text[:2000])
    return m.group(1) if m else os.path.splitext(filename)[0]

def clause_label(num):
    num = num.strip().rstrip(".")
    return re.sub(r"\s+", "", num) if num.startswith("제") else (num.rstrip(")") + "항")

def split_by_clause(lines, where=""):
    """줄 목록을 조항 제목 기준으로 묶습니다 → [(항목 번호, 제목, 글)]"""
    out, cur_no, cur_title, buf = [], None, "", []
    def flush():
        body = "\n".join(buf).strip()
        if body or cur_title:
            out.append((cur_no, cur_title, body))
    for ln in lines:
        m = CLAUSE.match(ln)
        if m and len(ln) <= 80:          # 너무 긴 줄은 번호로 시작하는 본문으로 봅니다
            flush()
            cur_no, cur_title, buf = clause_label(m.group(1)), (m.group(2) or "").strip(), []
        else:
            buf.append(ln)
    flush()
    return [(no or (f"머리말{where}" if where else "머리말"), t, b) for no, t, b in out]

# ---------- 2. 형식별 읽기 → 항목 단위 ----------
def read_docx(path):
    import docx
    d = docx.Document(path)
    lines = [p.text for p in d.paragraphs if p.text.strip()]
    for t in d.tables:
        for r in t.rows:
            lines.append(" | ".join(c.text.strip() for c in r.cells))
    return lines

def read_pdf(path):
    from pypdf import PdfReader
    lines = []
    for i, p in enumerate(PdfReader(path).pages, 1):
        for ln in (p.extract_text() or "").splitlines():
            if ln.strip():
                lines.append(ln)
        lines.append(f"[[{i}쪽 끝]]")
    return lines

def read_pptx(path):
    from pptx import Presentation
    lines = []
    for i, s in enumerate(Presentation(path).slides, 1):
        lines.append(f"{i}) 슬라이드 {i}")
        for sh in s.shapes:
            if sh.has_text_frame and sh.text_frame.text.strip():
                lines += [x for x in sh.text_frame.text.splitlines() if x.strip()]
    return lines

def read_hwpx(path):
    lines = []
    with zipfile.ZipFile(path) as z:
        for n in sorted(x for x in z.namelist() if re.match(r"Contents/section\d+\.xml", x)):
            xml = z.read(n).decode("utf-8", "ignore")
            for p in re.findall(r"<hp:p[ >].*?</hp:p>", xml, flags=re.S):
                t = html.unescape("".join(re.findall(r"<hp:t(?:\s[^>]*)?>(.*?)</hp:t>", p, flags=re.S)))
                t = re.sub(r"<[^>]+>", "", t)
                if t.strip():
                    lines.append(t)
    return lines

def read_text(path):
    for enc in ("utf-8", "cp949"):
        try:
            with open(path, encoding=enc) as f:
                return [x for x in f.read().splitlines() if x.strip()]
        except UnicodeDecodeError:
            continue
    raise ValueError("글자 인코딩을 알 수 없습니다")

def units_from_xlsx(path, name):
    """엑셀(사례집 · 목록): 행 하나가 항목 하나. 첫 칸이 번호 모양이면 그것이 항목 번호입니다"""
    import openpyxl
    wb = openpyxl.load_workbook(path, data_only=True, read_only=True)
    out = []
    for ws in wb.worksheets:
        rows = [["" if v is None else str(v).strip() for v in r] for r in ws.iter_rows(values_only=True)]
        rows = [r for r in rows if any(r)]
        if not rows:
            continue
        head, body = rows[0], rows[1:]
        for i, r in enumerate(body, 2):
            first = r[0]
            no = first if DOC_ID.fullmatch(first or "") or re.fullmatch(r"[A-Za-z0-9\-]{3,}", first or "") else f"{ws.title} {i}행"
            text = " · ".join(f"{h}: {v}" for h, v in zip(head, r) if v)
            # 열 이름(원인 · 조치 …)도 함께 찾습니다 — 「원인과 조치는?」처럼 열 이름으로 묻는 질문이 많아서
            out.append({"doc_id": doc_id_of(name, ""), "file": name, "clause": no, "title": ws.title,
                        "doc_title": os.path.splitext(name)[0], "text": text})
    return out

READERS = {".docx": read_docx, ".pdf": read_pdf, ".pptx": read_pptx, ".hwpx": read_hwpx,
           ".txt": read_text, ".md": read_text, ".csv": read_text}

def read_units(path):
    name = os.path.basename(path)
    ext = os.path.splitext(name)[1].lower()
    if ext == ".xlsx":
        return units_from_xlsx(path, name)
    lines = READERS[ext](path)
    did = doc_id_of(name, "\n".join(lines))
    doc_title = re.sub(r"\(.*?\)", "", lines[0]).strip() if lines else name   # 첫 줄 = 문서 제목
    out = []
    for no, title, body in split_by_clause(lines):
        # PDF: 「[[3쪽 끝]]」 표시로 그 항목이 어느 쪽에 있는지 적습니다
        pages = re.findall(r"\[\[(\d+)쪽 끝\]\]", body)
        body = re.sub(r"\n?\[\[\d+쪽 끝\]\]", "", body).strip()
        loc = no + (f" ({pages[0]}쪽)" if pages else "")
        full = (title + "\n" + body).strip()
        if no.startswith("머리말") and len(full) < 120:     # 제목 · 문서번호 · 개정 줄뿐이면 항목으로 두지 않습니다
            continue
        for k in range(0, max(len(full), 1), MAX_CHARS):
            piece = full[k:k + MAX_CHARS].strip()
            if piece:
                out.append({"doc_id": did, "file": name, "clause": loc, "title": title,
                            "doc_title": doc_title, "text": piece})
    return out

# ---------- 3. 개인정보 · 기밀 거르기 (검색 · AI 전에) ----------
SENSITIVE = ["개인정보", "주민등록", "주민번호", "연락처", "전화번호", "휴대폰", "집 주소", "주소지", "이메일 주소",
             "급여", "연봉", "월급", "인사평가", "인사 평가", "징계", "병력", "건강검진", "비밀번호", "계좌",
             "기밀", "대외비", "영업비밀", "단가", "원가", "납품가", "계약 금액", "계약금액", "고객 정보", "고객사 연락처"]
PII = re.compile(r"\d{6}-?[1-4]\d{6}|01[016789]-?\d{3,4}-?\d{4}|[\w.+-]+@[\w-]+\.[\w.]+")

def sensitive_reason(q):
    hit = [w for w in SENSITIVE if w in q.replace(" ", "") or w in q]
    if PII.search(q):
        hit.append("개인정보 모양(전화번호 · 주민번호 · 이메일)")
    return hit

# ---------- 4. 검색 (코랩 안에서) ----------
def search(units, q, k=TOP_K):
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    if not units or not q.strip():
        return [], 0.0
    # 항목마다 「문서 제목 · 번호 · 항목 제목」을 함께 넣어 찾습니다 — 「4.2 검사 기준」도 「수입검사 절차서」의 항목으로 잡히게
    docs = [f"{u.get('doc_title', '')} {u['doc_id']} {u['clause']} {u['title']}\n{u['text']}" for u in units]
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3), sublinear_tf=True)
    m = vec.fit_transform(docs + [q])
    score = cosine_similarity(m[-1], m[:-1])[0]
    best = sorted(range(len(units)), key=lambda i: -score[i])[:k]
    return [dict(units[i], score=float(score[i])) for i in best if score[i] > 0], float(score[best[0]])

# ---------- 5. AI 요약 ----------
SYSTEM = """너는 제조업 품질경영 문서를 찾아 주는 업무 도우미야. 아래 [문서 항목]만 근거로 답해.
[지켜야 할 것]
1. answer 는 3~5문장 요약. 질문에 바로 답하는 문장을 맨 앞에.
2. 문서에 없는 내용은 추측하거나 지어내지 마. 근거가 없으면 found 를 false 로.
3. sources 에는 근거로 쓴 항목을 적어. doc · clause 는 [문서 항목]의 <<문서번호 · 항목>> 을 그대로 쓰고, quote 는 원문을 한 글자도 바꾸지 말고 15~60자 옮겨.
4. 두 문서의 내용이 서로 다르면(숫자 · 기준 · 절차) conflicts 에 둘 다 적고, answer 에도 「문서마다 다릅니다」라고 밝혀. 어느 쪽이 맞는지 정하지 말고 개정판 · 상위 문서 확인을 권해.
5. 개인정보 · 기밀(연락처 · 급여 · 단가 · 원가 · 고객 정보 등)은 답하지 말고 담당 부서 확인을 안내해.
반드시 아래 JSON 하나로만 답해:
{"answer": "", "found": true,
 "sources": [{"doc": "", "clause": "", "quote": ""}],
 "conflicts": [{"topic": "", "a": {"doc": "", "clause": "", "says": ""}, "b": {"doc": "", "clause": "", "says": ""}}]}"""

def to_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "\n".join(c.get("text", "") if isinstance(c, dict) else str(c) for c in content)
    return str(content or "")

def friendly(e):
    s = str(e)
    if "invalid_api_key" in s or "Incorrect API key" in s:
        return "API 키가 맞지 않습니다. 키를 다시 확인해 주세요."
    if "insufficient_quota" in s:
        return "OpenAI 계정의 사용 한도(크레딧)가 없습니다. platform.openai.com 의 Billing 을 확인해 주세요."
    if "rate_limit" in s or "429" in s:
        return "요청이 너무 빨라 잠시 막혔습니다. 1분 뒤 다시 눌러 주세요."
    return "오류가 났습니다: " + s[:300]

def norm(s):
    return re.sub(r"\s+", "", s or "")

def quote_ok(units, src):
    """AI 가 적은 원문이 그 문서에 정말 있는지 — 문서는 「검사 절차서 SOP-QC-012」처럼 적어도 번호로 찾습니다"""
    q = norm(src.get("quote"))
    if len(q) < 5:
        return False
    doc = src.get("doc") or ""
    mine = [u for u in units if u["doc_id"] in doc or (doc and doc in u["file"])] or units
    return any(q in norm(u["text"]) for u in mine)

def sentences(t):
    return len([s for s in re.split(r"(?<=[.!?])\s+|\n+", (t or "").strip()) if s.strip()])

def render(units, data):
    if data.get("found") is False:
        return NOT_FOUND
    out = [(data.get("answer") or "").strip() or NOT_FOUND]
    cf = [c for c in (data.get("conflicts") or []) if isinstance(c, dict) and c.get("a") and c.get("b")]
    if cf:
        out.append("\n**⚠️ 문서 간 차이** — 어느 쪽이 맞는지는 개정판 · 상위 문서로 확인하세요.")
        for c in cf:
            a, b = c["a"], c["b"]
            out.append(f"- {c.get('topic', '')}: 「{a.get('doc')}, {a.get('clause')}」 {a.get('says', '')} ↔ "
                       f"「{b.get('doc')}, {b.get('clause')}」 {b.get('says', '')}")
    srcs = [s for s in (data.get("sources") or []) if isinstance(s, dict)]
    if srcs:
        out.append("\n**근거**")
        for s in srcs[:6]:
            mark = "" if quote_ok(units, s) else " ⚠️ 원문과 글자가 다릅니다 — 문서에서 직접 확인하세요"
            out.append(f"- 📄 「{s.get('doc', '')}, {s.get('clause', '')}」 — “{s.get('quote', '')}”{mark}")
    return "\n".join(out)

def extractive(hits):
    """키가 없을 때: 가까운 항목을 원문 그대로"""
    out = ["**OpenAI 키가 없어 요약 대신 가장 가까운 항목을 원문 그대로 보여 드립니다.**"]
    for h in hits[:3]:
        out.append(f"\n📄 「{h['doc_id']}, {h['clause']}」 {h['title']}\n> " + h["text"][:300].replace("\n", " "))
    out.append("\n_문서 간 차이 판단 · 요약은 키를 넣으면 합니다._")
    return "\n".join(out)

# ---------- 6. 질문하기 ----------
def ask(key_box, model, question, history, units):
    question = (question or "").strip()
    history = history or []
    if not question:
        raise gr.Error("질문을 적어 주세요.")
    if not units:
        raise gr.Error("먼저 왼쪽에서 문서를 읽어 주세요(「예제 문서로 연습하기」를 눌러도 됩니다).")
    reason = sensitive_reason(question)
    if reason:
        ans = (f"🔒 개인정보 · 기밀과 관련될 수 있는 질문입니다({', '.join(reason[:3])}).\n\n"
               f"이 챗봇은 답하지 않습니다. **{CONTACT}** 에 확인해 주세요. (질문은 검색 · AI 로 보내지 않았습니다)")
        return "", history + [{"role": "user", "content": question}, {"role": "assistant", "content": ans}]
    # 「그건?」 같은 이어진 질문도 찾도록 바로 앞 질문을 함께 넣어 찾습니다
    prev = next((to_text(m["content"]) for m in reversed(history) if m["role"] == "user"), "")
    hits, best = search(units, question + " " + prev)
    if best < MIN_SCORE:
        return "", history + [{"role": "user", "content": question}, {"role": "assistant", "content": NOT_FOUND}]
    key = (key_box or "").strip() or os.environ.get("OPENAI_API_KEY", "")
    if not key:
        ans = extractive(hits)
    else:
        from openai import OpenAI
        ctx = "\n\n".join(f"<<{h['doc_id']} · {h['clause']}>> ({h['file']}) {h['title']}\n{h['text']}" for h in hits)
        msgs = [{"role": "system", "content": SYSTEM}]
        for m in history[-4:]:
            msgs.append({"role": m["role"], "content": to_text(m["content"])[:2000]})
        msgs.append({"role": "user", "content": f"[문서 항목]\n{ctx}\n\n[질문]\n{question}"})
        try:
            r = OpenAI(api_key=key).chat.completions.create(
                model=model, messages=msgs, temperature=0.1, response_format={"type": "json_object"})
            raw = r.choices[0].message.content or "{}"
            data = json.loads(raw)
        except json.JSONDecodeError:
            data = {"answer": raw, "found": True}
        except Exception as e:
            raise gr.Error(friendly(e))
        ans = render(units, data)
        n = sentences(data.get("answer") or "")
        if data.get("found") is not False and not 3 <= n <= 5:
            ans += f"\n\n_(요약이 {n}문장입니다 — 요청 기준 3~5문장)_"
    return "", history + [{"role": "user", "content": question}, {"role": "assistant", "content": ans}]

# ---------- 7. 문서 읽기 ----------
def doc_table(units):
    if not units:
        return "아직 읽은 문서가 없습니다."
    rows = ["| 문서 번호 | 파일 | 항목 수 | 항목 예 |", "|---|---|---|---|"]
    seen = {}
    for u in units:
        seen.setdefault((u["doc_id"], u["file"]), []).append(u["clause"])
    for (d, f), cl in seen.items():
        ex = ", ".join(dict.fromkeys(cl))
        rows.append(f"| {d} | {f} | {len(set(cl))} | {ex[:60]}{'…' if len(ex) > 60 else ''} |")
    return "\n".join(rows)

def load(paths, units):
    msgs = []
    units = []          # 다시 읽으면 처음부터 새로
    for p in paths or []:
        p = p if isinstance(p, str) else p.name
        name = os.path.basename(p)
        ext = os.path.splitext(name)[1].lower()
        if ext in (".hwp", ".doc", ".ppt", ".xls"):
            msgs.append(f"❌ {name}: 옛 형식은 읽지 못합니다. HWPX · DOCX · PPTX · XLSX · PDF 로 저장해 올려 주세요.")
            continue
        if ext not in READERS and ext != ".xlsx":
            msgs.append(f"❌ {name}: 지원하지 않는 형식입니다(pdf · docx · xlsx · pptx · hwpx · txt).")
            continue
        try:
            u = read_units(p)
        except Exception as e:
            msgs.append(f"❌ {name}: 읽기 오류 — {e}")
            continue
        if not u:
            msgs.append(f"⚠️ {name}: 글자가 없습니다(스캔 PDF 일 수 있습니다).")
            continue
        units += u
        no_clause = all(x["clause"].startswith(("머리말", "문단")) for x in u)
        msgs.append(f"✅ {name}: 문서 번호 {u[0]['doc_id']} · 항목 {len(set(x['clause'] for x in u))}개"
                    + (" — 조항 번호를 찾지 못해 위치로 표시합니다" if no_clause else ""))
    return units, "\n\n".join(msgs) or "파일을 올려 주세요.", doc_table(units)

def load_upload(files, units):
    return load(files, units)

def load_folder(folder, units):
    folder = (folder or "").strip()
    if not os.path.isdir(folder):
        raise gr.Error(f"폴더를 찾을 수 없습니다: {folder}")
    names = sorted(n for n in os.listdir(folder) if not n.startswith("."))
    return load([os.path.join(folder, n) for n in names], units)

def load_samples(units):
    return load_folder(os.environ.get("SAMPLE_DIR", "/content/품질문서_예제"), units)

# ---------- 8. 화면 ----------
HOWTO = """**시작하는 방법** ① 왼쪽에서 「예제 문서로 연습하기」를 누르거나 내 문서를 올리고 「문서 읽기」 →
② 오른쪽에 질문을 적고 Enter → ③ 답 아래 **근거(문서 번호, 항목)** 를 확인합니다. 문서끼리 다르면 「⚠️ 문서 간 차이」가 나옵니다."""

chat_kwargs = {"label": "품질 자료 질문", "height": 480}
if "type" in inspect.signature(gr.Chatbot.__init__).parameters:     # Gradio 4~5
    chat_kwargs["type"] = "messages"

with gr.Blocks(title="품질 자료 검색 · 요약 챗봇") as demo:
    units = gr.State([])
    gr.Markdown("# 🔎 품질 자료 검색 · 요약 챗봇")
    gr.Markdown(HOWTO)
    with gr.Row():
        key_box = gr.Textbox(label="OpenAI API 키 (보안 비밀에 넣었다면 비워 두기 · 없으면 원문 항목만)", type="password", scale=3)
        model = gr.Dropdown(["gpt-4.1-mini", "gpt-4o-mini", "gpt-4.1"], value="gpt-4.1-mini", label="모델",
                            allow_custom_value=True, scale=1)
    with gr.Row():
        with gr.Column(scale=1, min_width=300):
            sample_btn = gr.Button("예제 문서로 연습하기", variant="secondary")
            files = gr.File(label="품질 문서 올리기", file_count="multiple", type="filepath")
            load_btn = gr.Button("문서 읽기", variant="primary")
            with gr.Accordion("구글 드라이브 폴더에서 읽기", open=False):
                folder = gr.Textbox(label="폴더 경로", value=os.environ.get("DOC_FOLDER", ""))
                folder_btn = gr.Button("폴더 읽기")
            msg = gr.Markdown()
            table = gr.Markdown("아직 읽은 문서가 없습니다.")
        with gr.Column(scale=2):
            chat = gr.Chatbot(**chat_kwargs)
            q = gr.Textbox(label="질문", placeholder="예: 수입검사 시료는 몇 개 뽑아?")
            gr.Examples(["수입검사 시료는 몇 개 뽑아?", "공정검사 기준은?", "작년 버 불량 원인과 조치는?",
                         "부적합품은 어떻게 처리해?", "공급사 담당자 연락처 알려줘"], q)
            clear = gr.Button("대화 지우기", size="sm")

    outs = [units, msg, table]
    sample_btn.click(load_samples, units, outs)
    load_btn.click(load_upload, [files, units], outs)
    folder_btn.click(load_folder, [folder, units], outs)
    q.submit(ask, [key_box, model, q, chat, units], [q, chat])
    clear.click(lambda: [], None, chat)

if __name__ == "__main__":
    demo.launch(share=True, debug=True)

## 해 보기 (예제 문서 기준)
| 질문 | 확인할 것 |
|---|---|
| 수입검사 시료는 몇 개 뽑아? | **⚠️ 문서 간 차이** — QM-001 7.1항(5개) ↔ SOP-QC-012 4.2항(로트 크기별 8 · 13 · 20개) |
| 작년 버 불량 원인과 조치는? | 사례집 NC-2025-003 · NC-2025-014 — 원인(금형 날 마모)과 조치 |
| 초물 검사는 언제 해? | SOP-QC-015 5.1항 |
| 부적합품은 어떻게 처리해? | QM-001 8.3항(격리 · 24시간 안 보고서) |
| 오늘 점심 메뉴 추천해줘 | 「확인되지 않습니다」(AI 를 부르지 않음) |
| 공급사 담당자 연락처 알려줘 | 🔒 담당 부서 확인 안내(검색 · AI 를 하지 않음) |

## 내 문서로 바꿀 때
- 조항 제목은 **줄 맨 앞**에 「4.2 검사 기준」 · 「제5조(목적)」처럼 있어야 항목 번호로 읽힙니다. 왼쪽 표의 「항목 예」에서 확인하세요.
- 문서 번호는 파일 이름에 넣으면 가장 확실합니다(예: `검사절차서_SOP-QC-012.docx`).
- 검색은 글자 조각을 비교합니다. 「버」 · 「핀」 같은 한 글자 낱말은 문서에 「버(burr)」처럼 붙여 쓰면 잘 안 잡히니 「버 과다(burr)」처럼 띄어 쓴 표기를 함께 두세요.
- 담당 부서 이름은 ⑤ 셀 맨 위 `CONTACT` 를 바꿉니다. 거를 낱말은 `SENSITIVE` 목록에 더합니다.

## 오류가 나면
| 증상 | 원인 | 해결 |
|---|---|---|
| 항목 번호가 「머리말」 · 「문단」으로만 나온다 | 조항 제목이 줄 맨 앞에 없거나 표 안에 있다 | 문서에서 조항 제목 줄을 확인, 또는 PDF 대신 원본 DOCX 로 |
| 「확인되지 않습니다」가 너무 자주 나온다 | 문서에 쓰는 낱말과 질문 낱말이 다르다 | 문서의 낱말로 물어보기(「시료」 · 「샘플링」), 필요하면 `MIN_SCORE` 를 0.02 로 |
| 질문이 🔒 로 막힌다 | `SENSITIVE` 낱말이 들어 있다 | 업무에 꼭 필요한 낱말이면 목록에서 빼기(회사 규정 확인 뒤) |
| 「사용 한도(크레딧)가 없습니다」 | OpenAI 결제 · 크레딧 | platform.openai.com → Billing, 또는 키 없이 원문 항목으로 |
| `Data incompatible with messages format` | 옛 Gradio 코드 | 이 노트북은 Gradio 6 형식(role · content) — ⑤ 셀을 다시 실행 |